# 2.1 Frequency Distribution

**Goal:** turn raw observations into interpretable frequency tables without losing track of denominators, order, and interval boundaries.

**Prerequisites:** Module 1 data types and measurement scales. **Runtime:** Python 3.10+, standard library only. Examples are synthetic and outputs are saved. Run cells in order or use **Restart Kernel and Run All**.

## Contents
1. Frequency
2. Relative frequency
3. Cumulative frequency
4. Frequency tables
5. Grouped and ungrouped data
6. Class intervals
7. EDA application, practice, and solutions

## 1. Frequency

The **frequency** $f_j$ of a value or category is its number of occurrences. For mutually exclusive categories covering all $n$ observed units, $\sum_j f_j=n$. Counts refer to a defined unit: counting sessions is different from counting distinct users.

**Worked example:** daily support-ticket counts of $0,1,1,2,2,2,3,4$ contain three occurrences of 2, so the frequency at 2 is 3. The frequency of a value not observed is zero, not a missing measurement.

For a categorical variable, categories may be nominal (browser type) or ordered (low/medium/high). Preserve meaningful order when summarizing ordered categories; alphabetic order can be misleading.

In [1]:
from collections import Counter
import math
counts_per_day = [0, 1, 1, 2, 2, 2, 3, 4]
n = len(counts_per_day)
frequencies = Counter(counts_per_day)
print("Frequency of 2:", frequencies[2])
print("Frequency of 5:", frequencies[5])
print("Sum of frequencies:", sum(frequencies.values()))
assert sum(frequencies.values()) == n

Frequency of 2: 3
Frequency of 5: 0
Sum of frequencies: 8


## 2. Relative Frequency

Relative frequency is $r_j=f_j/n$, and percentage frequency is $100r_j\%$. For exhaustive categories and a common denominator, relative frequencies sum to 1, apart from rounding.

**Worked example:** the relative frequency of 2 tickets is $3/8=0.375$, or 37.5%. This describes the observed eight days. Treating 0.375 as the probability of 2 tickets on a future day requires additional assumptions about the process and sample.

Always state the denominator. If 2 of 10 survey responses are missing, 4 “yes” answers represent 40% of all records but 50% of the 8 nonmissing responses. Neither denominator should be hidden. Do not add percentages with different denominators as though they were raw counts.

In [2]:
responses = ["yes"]*4 + ["no"]*4 + [None]*2
observed = [r for r in responses if r is not None]
print("Yes among all records:", responses.count("yes")/len(responses))
print("Yes among observed responses:", observed.count("yes")/len(observed))
print("Missing response count:", len(responses)-len(observed))

Yes among all records: 0.4
Yes among observed responses: 0.5
Missing response count: 2


## 3. Cumulative Frequency

For sorted values or ordered classes, cumulative frequency is $F_j=\sum_{k\le j}f_k$: the number at or below the current value/class upper boundary according to the stated inclusion convention. Cumulative relative frequency is $R_j=F_j/n$.

**Worked example:** the frequencies for $0,1,2,3,4$ are $1,2,3,1,1$, so cumulative frequencies are $1,3,6,7,8$. Six of eight days had **at most** 2 tickets. The proportion with more than 2 is $1-6/8=0.25$.

Cumulative frequencies should never decrease and should finish at the number of observations counted. They have substantive meaning for numerical and ordinal variables, but not for arbitrary orders of nominal categories such as browser names.

## 4. Frequency Tables

A useful table includes the value or class, frequency, relative frequency, and (when meaningful) cumulative columns. Keep sufficient precision internally; displayed rounded percentages can sum to 99.9% or 100.1%.

In [3]:
cumulative = 0
print(f"{'tickets':>7} {'freq':>5} {'relative':>9} {'cum.freq':>9} {'cum.rel':>9}")
for value in sorted(frequencies):
    f = frequencies[value]
    cumulative += f
    print(f"{value:7} {f:5} {f/n:9.3f} {cumulative:9} {cumulative/n:9.3f}")
assert cumulative == n
assert math.isclose(sum(f/n for f in frequencies.values()), 1)
print("Proportion above 2:", sum(f for x, f in frequencies.items() if x > 2)/n)

tickets  freq  relative  cum.freq   cum.rel
      0     1     0.125         1     0.125
      1     2     0.250         3     0.375
      2     3     0.375         6     0.750
      3     1     0.125         7     0.875
      4     1     0.125         8     1.000
Proportion above 2: 0.25


## 5. Grouped and Ungrouped Data

**Ungrouped data** retain individual observations or counts at each exact observed value. They allow exact calculations from the original values. **Grouped data** combine values into classes such as durations from 0 to under 10 seconds. Grouping simplifies a distribution but loses within-class detail.

**Worked example:** durations $2,5,7,12,16,18,23,28,30$ can be grouped into 0–<10, 10–<20, and 20–30. These contain 3, 3, and 3 observations. The table is compact, but it cannot reveal which exact durations occurred within each class.

If only grouped counts and class midpoints $m_j$ are known, an approximate mean is
$$\bar x_{\text{grouped}}\approx\frac{\sum_j f_jm_j}{\sum_j f_j}.$$
This substitutes each midpoint for every class member. It is not generally the exact raw-data mean. Open-ended classes such as “100 or more” do not have a finite midpoint without further assumptions.

## 6. Class Intervals

A class interval specifies a lower and upper boundary. Class width is upper minus lower boundary; midpoint is their average. Adjacent classes should cover the intended domain without gaps or double counting.

We use **left-closed, right-open** classes $[a,b)$, except the last class is closed on the right. Thus 10 belongs to $[10,20)$, not $[0,10)$, and 30 is included in the final $[20,30]$ class. Other conventions are valid if stated consistently.

For integer measurements rounded to the nearest whole unit, apparent class limits 10–19 may correspond to real boundaries 9.5–19.5. Boundaries should reflect the measurement precision rather than an arbitrary correction applied everywhere.

Choosing bins changes the visible distribution: very wide bins hide structure; very narrow bins make random noise prominent. Equal widths simplify reading. With unequal widths, a density histogram uses height $f_j/(n\times\text{width}_j)$ so **area**, not height alone, represents relative frequency. There is no universally best bin count for every question.

In [4]:
durations = [2, 5, 7, 12, 16, 18, 23, 28, 30]
edges = [0, 10, 20, 30]
def grouped_counts(values, edges):
    if len(edges) < 2 or any(b <= a for a, b in zip(edges, edges[1:])):
        raise ValueError("Edges must be strictly increasing")
    result = [0]*(len(edges)-1)
    for value in values:
        if not math.isfinite(value) or not edges[0] <= value <= edges[-1]:
            raise ValueError("Value outside finite class domain")
        for j, (a, b) in enumerate(zip(edges, edges[1:])):
            if a <= value < b or (j == len(result)-1 and value == b):
                result[j] += 1
                break
    return result

freq = grouped_counts(durations, edges)
for j, (a, b, f) in enumerate(zip(edges, edges[1:], freq)):
    closing = "]" if j == len(freq)-1 else ")"
    print(f"[{a}, {b}{closing}: frequency={f}, midpoint={(a+b)/2}, width={b-a}")
midpoints = [(a+b)/2 for a, b in zip(edges, edges[1:])]
print("Raw-data mean:", sum(durations)/len(durations))
print("Midpoint approximate mean:", sum(f*m for f, m in zip(freq, midpoints))/len(durations))
assert freq == [3, 3, 3]
assert grouped_counts([0, 10, 20, 30], edges) == [1, 1, 2]

[0, 10): frequency=3, midpoint=5.0, width=10
[10, 20): frequency=3, midpoint=15.0, width=10
[20, 30]: frequency=3, midpoint=25.0, width=10
Raw-data mean: 15.666666666666666
Midpoint approximate mean: 15.0


## 7. AI/ML application: inspect class imbalance

Frequency tables reveal rare target labels and missing or unexpected categories before training. A model predicting only the largest class can look accurate on an imbalanced dataset while being useless for the minority class. Training and evaluation should examine class-specific metrics, not only total accuracy.

Compare distributions across train and deployment periods using aligned categories and clearly defined denominators. A changed frequency can indicate a collection change or distribution shift; it does not identify the cause by itself.

In [5]:
labels = ["routine"]*90 + ["urgent"]*10
label_counts = Counter(labels)
print("Label counts:", dict(label_counts))
majority_accuracy = max(label_counts.values())/len(labels)
print("Majority-only baseline accuracy:", majority_accuracy)
print("Urgent cases found by always predicting routine:", 0)

Label counts: {'routine': 90, 'urgent': 10}
Majority-only baseline accuracy: 0.9
Urgent cases found by always predicting routine: 0


## Practice

1. Make a frequency/relative-frequency table for $1,1,2,3,3,3,4,4$.
2. How many values are at most 3? What proportion are greater than 3?
3. Of 50 survey records, 10 are missing and 24 say yes. Find the yes proportion among all records and among nonmissing responses.
4. Group $0,5,10,15,20$ into $[0,10)$ and $[10,20]$.
5. Classes $[0,2)$ and $[2,6]$ each contain 4 of 8 observations. Find relative frequencies and density-histogram heights.
6. Why is a midpoint-based mean usually approximate?
7. Is cumulative frequency meaningful after sorting unordered product brands alphabetically?

## Solutions

1. Values 1,2,3,4 have frequencies 2,1,3,2 and relative frequencies 0.25,0.125,0.375,0.25.
2. Cumulative count at 3 is 6; proportion greater than 3 is $2/8=0.25$.
3. $24/50=0.48$ and $24/40=0.60$. Report which denominator you use.
4. Frequencies 2 and 3. The shared boundary 10 belongs only to the second class.
5. Relative frequencies are 0.5 each; widths 2 and 4 give heights 0.25 and 0.125. Each bar's area is 0.5.
6. It replaces each true value with a class midpoint. The original deviations within the class are unknown and may not cancel.
7. Not substantively: alphabetic order is arbitrary, so cumulative counts do not describe a natural “at or below” quantity.

**Readiness check:** audit denominators, verify total counts, define boundary inclusion, and distinguish histogram height from probability area.

**Next:** [2.2 Measures of Central Tendency](2_Measures-of-Central-Tendency.ipynb).